# Exploratory Data Analysis (EDA)
## Occupancy Prediction for Antalya/Alanya Resort Hotels Using Google Trends

**Prepared by:** Bedirhan Sar  
**Project:** Hotel Occupancy Analysis and Prediction

---

## Executive Summary

This study evaluates whether Google Trends can provide useful external signal for understanding and later predicting **daily hotel occupancy rates** for two resort hotels in the Antalya/Alanya region of Turkey.

The analysis combines:
- hotel occupancy data from **Side Mare Hotel** and **Azura Deluxe**,
- Google Trends data from **Germany, the Netherlands, the United Kingdom, and Türkiye**.

The main result of the exploratory phase is that **same-day Google Trends relationships are generally weak to moderate**, while **several lagged Google Trends features show stronger associations with occupancy**. This suggests that search behavior is more useful as an **early signal** than as a same-day explanatory variable.


## Project Motivation

Resort hotel demand is shaped by strong seasonality, travel planning cycles, destination interest, and B2B distribution channels such as agencies and tour operators. Because of this market structure, Google Trends should not be treated as a direct measure of bookings. Instead, it is better interpreted as a possible upstream indicator of travel intent.

The main research question is:

**Can Google Trends data provide useful information about future occupancy patterns for Antalya/Alanya resort hotels?**


## Data Sources

### Hotel Data
Daily occupancy data was prepared for:
- **Side Mare Hotel**
- **Azura Deluxe**

Primary target variable:
- `occupancy_rate`

### Google Trends Data
Tourism-related Google Trends data was collected for:
- **Germany**
- **Netherlands**
- **United Kingdom**
- **Türkiye**

The raw Trends data was initially organized around:
- date,
- country,
- keyword,
- trend score.


## Data Preparation Workflow

The analytical dataset was built in four steps:

1. **Hotel standardization**  
   Common schema:
   - `date`
   - `hotel_name`
   - `occupancy_rate`

2. **Google Trends standardization**  
   Common schema:
   - `date`
   - `country`
   - `keyword`
   - `google_trends_score`

3. **Pivoting to wide format**  
   Trend variables were converted into columns such as:
   - `trends_germany_alanya_hotel`
   - `trends_turkiye_side_otel`
   - `trends_netherlands_hotel_alanya`

4. **Building the master table**  
   The hotel data and trend data were merged by `date`, producing a dataset where:

   **one row = one hotel on one date**


## Master Table Overview

| Metric | Value |
|---|---:|
| Total rows | 1307 |
| Total columns | 19 |
| Hotels | 2 |
| Date range | 2023-03-25 to 2025-11-08 |
| Duplicate `(date, hotel_name)` rows | 0 |
| Trends missing values per feature | 78 |


In [ ]:
import pandas as pd

master = pd.read_excel("hotel_master_table.xlsx", sheet_name="master_table")
master["date"] = pd.to_datetime(master["date"])
master.head()


## Data Quality Assessment

The merged dataset was reviewed for:
- data type consistency,
- duplicate rows,
- missing values,
- and date-range overlap.

### Summary assessment
The dataset was suitable for analysis. No duplicate `(date, hotel_name)` rows were found. Missing values were concentrated in the Trends variables and were mainly caused by partial mismatch between hotel coverage and Trends collection windows.


In [ ]:
master.info()
master.isna().sum().sort_values(ascending=False).head(20)


## Research Hypotheses

### H1
Google Trends features have a measurable relationship with hotel occupancy.

### H2
Lagged Google Trends values are more informative than same-day Google Trends values.

### H3
The strength of the relationship varies across country-keyword combinations.

### H4
Occupancy exhibits strong seasonal structure, which must be taken into account when interpreting external signals.


## Occupancy Summary by Hotel

| Hotel | Count | Mean | Median | Std. Dev. | Min | Max |
|---|---:|---:|---:|---:|---:|---:|
| Azura Deluxe | 694 | 82.5533 | 94.385 | 23.0493 | 0.00 | 103.56 |
| Side Mare Hotel | 613 | 93.2150 | 97.813 | 14.9361 | 0.00 | 103.75 |

### Interpretation
- Side Mare Hotel has the higher average occupancy level.
- Azura Deluxe shows larger variability.
- Both hotels display strong seasonality.
- Near-zero values are present in low-demand or likely off-season periods.


In [ ]:
master.groupby("hotel_name")["occupancy_rate"].agg(["count", "mean", "median", "std", "min", "max"])


## Visualization 1: Daily Occupancy Over Time

![Daily Occupancy Over Time](Visualizations/occupancy_over_time.png)

This figure highlights the strong seasonal structure of the data and shows that the two hotels follow broadly similar annual rhythms, while still displaying different levels of volatility.


## Visualization 2: Monthly Average Occupancy

![Monthly Average Occupancy](Visualizations/monthly_occupancy.png)

The monthly view makes the seasonality easier to read by smoothing daily noise. The repeated annual pattern confirms that occupancy is strongly seasonal in both hotels.


## Same-Day Correlation Analysis

The first statistical comparison measured whether same-day Google Trends values were associated with same-day occupancy.

### Top Same-Day Pearson Correlations

| Feature | Pearson r | p-value |
|---|---:|---:|
| trends_turkiye_alanya_otel | 0.282470 | 5.58934e-24 |
| trends_turkiye_side_otel | 0.249920 | 5.89086e-19 |
| trends_germany_alanya_hotel | 0.239346 | 1.79783e-17 |
| trends_turkiye_alanya_otel_fiyatlari | 0.219890 | 6.36611e-15 |
| trends_netherlands_hotel_alanya | 0.148717 | 1.62815e-07 |

![Top Same-Day Correlations](Visualizations/top_same_day_correlations.png)

### Interpretation
The same-day relationships are generally weak to moderate. This is consistent with the market structure of the hotels: occupancy is influenced by booking pipelines and agency-driven flows, so same-day search activity is not expected to fully explain same-day stays.


In [ ]:
from scipy.stats import pearsonr, spearmanr

trend_cols = [c for c in master.columns if c.startswith("trends_")]

rows = []
for c in trend_cols:
    sub = master[["occupancy_rate", c]].dropna()
    if len(sub) > 10 and sub[c].nunique() > 1:
        p = pearsonr(sub["occupancy_rate"], sub[c])
        s = spearmanr(sub["occupancy_rate"], sub[c], nan_policy="omit")
        rows.append({
            "feature": c,
            "pearson_r": p.statistic,
            "pearson_p": p.pvalue,
            "spearman_rho": s.statistic,
            "spearman_p": s.pvalue,
        })

same_day_corr = pd.DataFrame(rows).sort_values("pearson_r", ascending=False)
same_day_corr.head(10)


## Lag Analysis

Because travel-related search behavior may occur days or weeks before an actual hotel stay, lagged Trends features were examined at 7, 14, 21, and 28 days.

### Top Best Lagged Pearson Correlations

| Feature | Best Lag (days) | Pearson r | p-value |
|---|---:|---:|---:|
| trends_turkiye_side_otel | 28 | 0.399829 | 1.16762e-44 |
| trends_germany_alanya_hotel | 7 | 0.255002 | 2.51628e-19 |
| trends_turkiye_alanya_otel_fiyatlari | 21 | 0.229107 | 3.47173e-15 |
| trends_netherlands_hotel_alanya | 28 | 0.207813 | 1.68567e-12 |
| trends_turkiye_antalya_tatil | 28 | 0.191652 | 8.11092e-11 |

![Top Lagged Correlations](Visualizations/top_lagged_correlations.png)

### Interpretation
The lag analysis provides one of the most important results of the exploratory phase: several Trends variables become more informative when shifted backward in time. This supports the idea that search interest is more relevant as an **early indicator** than as an immediate signal.


In [ ]:
trend_df = master[["date"] + trend_cols].drop_duplicates("date").sort_values("date").reset_index(drop=True)

lag_rows = []
for lag in [7, 14, 21, 28]:
    lagged = trend_df.copy()
    lagged[trend_cols] = lagged[trend_cols].shift(lag)
    merged = master[["date", "hotel_name", "occupancy_rate"]].merge(lagged, on="date", how="left")
    for c in trend_cols:
        sub = merged[["occupancy_rate", c]].dropna()
        if len(sub) > 10 and sub[c].nunique() > 1:
            p = pearsonr(sub["occupancy_rate"], sub[c])
            lag_rows.append({
                "feature": c,
                "lag_days": lag,
                "pearson_r": p.statistic,
                "pearson_p": p.pvalue,
            })

lag_corr = pd.DataFrame(lag_rows)
best_by_feature = lag_corr.loc[lag_corr.groupby("feature")["pearson_r"].idxmax()].sort_values("pearson_r", ascending=False)
best_by_feature.head(10)


## Visualization 3: Best Lagged Signal Overlay

![Best Lag Overlay](Visualizations/best_lag_overlay.png)

This overlay compares normalized occupancy with the strongest lagged Google Trends signal. It does not prove causality, but it provides visual support for the idea that delayed search behavior may move in a pattern that is more aligned with occupancy than same-day search behavior.


## Formal Hypothesis Assessment

| Hypothesis | Assessment | Summary |
|---|---|---|
| H1: Google Trends features have a measurable relationship with hotel occupancy. | **Partially supported** | Some country-keyword combinations show meaningful association, but the effect is not uniformly strong across all variables. |
| H2: Lagged Google Trends values are more informative than same-day values. | **Supported** | Several lagged features exhibit stronger correlations than their same-day counterparts. |
| H3: Relationship strength varies by country and keyword. | **Supported** | Search terms from Türkiye and Germany appear stronger than many of the United Kingdom features. |
| H4: Occupancy has strong seasonal structure. | **Supported** | Occupancy displays a clear seasonal pattern in both daily and monthly visualizations. |


## Key Analytical Conclusions

1. **Seasonality is the dominant structural pattern** in the occupancy data.
2. **Same-day Google Trends effects are limited**, and should not be overstated.
3. **Lagged Google Trends signals are more promising**, especially for selected country-keyword pairs.
4. **Not all search features are equally useful**, so feature selection is necessary before modeling.
5. Google Trends should be interpreted as a **supporting explanatory layer**, not as a complete representation of hotel demand.


## Limitations

1. **Google Trends is not booking data**  
   It measures search interest, not confirmed reservations.

2. **B2B demand channels are important**  
   Tour operators and agencies likely explain part of occupancy behavior that search data does not capture.

3. **Correlation does not establish causation**  
   These relationships identify association, not direct causal mechanisms.

4. **Seasonality can inflate apparent relationships**  
   Since both Trends and occupancy may rise during the same periods, part of the observed relationship may reflect shared seasonality.

5. **Feature quality differs across keywords and countries**  
   Some search features are informative, while others may mostly contribute noise.


## Next Step

The next stage is to convert the most promising Trends variables into lagged model features and combine them with:
- calendar information,
- hotel identity,
- and past occupancy values.

That modeling stage will test whether the exploratory relationships observed here translate into measurable predictive improvement.


## Final Summary

This exploratory analysis established a structured foundation for the predictive phase of the project.

In summary:
- two hotel occupancy datasets were cleaned and combined,
- four-country Google Trends data was collected and integrated,
- a master analytical table was built,
- data quality checks were completed,
- seasonal occupancy structure was documented,
- same-day and lagged statistical relationships were tested,
- and lagged search features emerged as the most promising external signals.

Overall, the evidence suggests that Google Trends is not sufficient on its own to explain resort hotel occupancy, but selected lagged search features may provide useful incremental information within a broader forecasting framework.
